# UAVIDS-2025 — baselines preditivos sem GNN

Este notebook apresenta a rodada exploratória `predictive_baseline_v1`. Os modelos foram executados fora do notebook por um módulo retomável; aqui carregamos somente configurações, predições e métricas persistidas. Isso permite recalcular as tabelas sem retreinar.

**Questões desta rodada:** (1) o controle de assinaturas repetidas reduz fortemente o resultado? (2) separar origens muda a estabilidade? (3) árvores, boosting ou MLP compacta oferecem a melhor fronteira inicial? (4) qual é o efeito diagnóstico de `FlowID`?


## Protocolo e fontes metodológicas

O [protocolo v1](../../protocol/evaluation_v1.md) foi escrito antes da execução. Pipelines e validação externa seguem as recomendações de prevenção de vazamento e grupos do [scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html) e sua [documentação de cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).

A comparação com `FlowID` foi motivada pela ambiguidade de R7 entre remoção de identificadores e a importância atribuída a essa coluna ([artigo](https://doi.org/10.1186/s13635-026-00234-w)). Ela é marcada como diagnóstica. A MLP compacta testa uma primeira parte de E1; não é apresentada como reprodução da CNN residual de [R4](https://www.nature.com/articles/s41598-026-52524-5).


In [1]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent

result_dir = project_root / "results" / "predictive_baseline_v1"
report_dir = project_root / "reports" / "predictive_baseline_v1"
manifest = json.loads((result_dir / "experiment_manifest.json").read_text("utf-8"))
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "experiment_id": "predictive_baseline_v1",
  "status": "exploratory_baseline_not_confirmatory",
  "config_sha256": "c9c4c015bf183f934fb563216eb455696903ed02ce70d68dc543fc14aa04e142",
  "dataset_sha256": "d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0",
  "split_sha256": "352d1966e2dda8060a7a59d69eafa49c461e4c2415db292319e0e1526d2996cc",
  "completed_jobs": 105,
  "expected_jobs": 105,
  "complete": true,
  "environment": {
    "python": "3.12.10",
    "platform": "Windows-11-10.0.26200-SP0",
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "scikit_learn": "1.5.2",
    "xgboost": "2.0.3",
    "processor": "AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD",
    "logical_cpu_count": 12,
    "threads_allowed_per_fit": 4,
    "accelerator": "none_used"
  },
  "class_order": [
    "Normal Traffic",
    "Blackhole Attack",
    "Flooding Attack",
    "Sybil Attack",
    "Wormhole Attack"
  ],
  "prediction_schema": [
    "row_id",
    "flow_id",
    "protocol",
    "fol

## Integridade da execução

Um job corresponde a um modelo em um fold e protocolo. Cada arquivo de predição tem checksum próprio; o manifesto liga a execução aos hashes dos dados, splits e configuração.


In [2]:
metrics_by_fold = pd.read_csv(result_dir / "metrics_by_fold.csv")
pooled_metrics = pd.read_csv(result_dir / "metrics_pooled_oof.csv")
class_metrics = pd.read_csv(result_dir / "class_metrics_pooled_oof.csv")

print(f"Jobs registrados: {len(metrics_by_fold)}")
print(f"Combinações completas protocolo/modelo: {len(pooled_metrics)}")
print(f"Falhas: {len(list((result_dir / 'job_records').glob('*failed.json')))}")


Jobs registrados: 105
Combinações completas protocolo/modelo: 21
Falhas: 0


## Qualidade preditiva sem identificadores

In [3]:
main_columns = [
    "protocol", "model", "f1_macro", "accuracy", "log_loss",
    "false_alarm_rate", "missed_attack_rate",
]
main_results = pooled_metrics.loc[~pooled_metrics["diagnostic_only"], main_columns]
print(main_results.round(6).to_string(index=False))


protocol               model  f1_macro  accuracy  log_loss  false_alarm_rate  missed_attack_rate
      S0         dummy_prior  0.070572  0.214224  1.604038          0.000000            1.000000
      S0         extra_trees  0.952558  0.949653  0.162444          0.011615            0.003510
      S0 logistic_regression  0.817804  0.815824  0.543921          0.092733            0.028677
      S0         mlp_compact  0.935207  0.933372  0.153735          0.020327            0.004750
      S0       random_forest  0.955051  0.952403  0.112881          0.009476            0.003063
      S0             xgboost  0.956326  0.954064  0.100617          0.008291            0.002917
      S1         dummy_prior  0.070572  0.214224  1.604052          0.000000            1.000000
      S1         extra_trees  0.951624  0.948818  0.167560          0.011768            0.003281
      S1 logistic_regression  0.818158  0.816168  0.542346          0.092389            0.028657
      S1         mlp_compact  

![F1 por protocolo](../../reports/predictive_baseline_v1/f1_by_protocol.png)

As barras representam resultados dos folds; a dispersão não deve ser interpretada como replicação em datasets independentes. S2 possui origens inteiras e folds de tamanhos diferentes.


In [4]:
fold_summary = pd.read_csv(report_dir / "model_comparison.csv")
print(fold_summary.to_string(index=False))


protocol         model  diagnostic_only  f1_macro__mean  f1_macro__std  fit_seconds__mean  inference_microseconds_per_row_amortized__mean  serialized_model_mib__mean
      S0         Dummy            False        0.070572       0.000005           0.002862                                        0.024904                    0.000896
      S0   Extra Trees            False        0.952558       0.001852           3.232619                                        7.467597                  491.295750
      S0      Logistic            False        0.817802       0.003846           5.447640                                        0.308676                    0.002512
      S0  MLP compacta            False        0.935200       0.001617          38.535585                                        0.669451                    0.086515
      S0 Random Forest            False        0.955050       0.001608          13.108422                                        4.225050                  167.860299
    

## Ablação de `FlowID`

Em S0, RF sem identificador obteve F1-macro OOF de **0.955051**. Com `FlowID`, chegou a **0.999668**. A diferença mostra que a coluna codifica fortemente a organização/rótulo do arquivo. O resultado com identificador não participa do ranking principal.


In [5]:
flow_id_ablation = pd.read_csv(report_dir / "flow_id_ablation.csv")
print(flow_id_ablation.to_string(index=False))


protocol  random_forest  random_forest_with_flow_id  absolute_f1_increase
      S0       0.955051                    0.999668              0.044617
      S1       0.953986                    0.999648              0.045662
      S2       0.950677                    0.999665              0.048988


## Classes difíceis e hipótese contrária

Se as repetições exatas explicassem a maior parte do resultado, seria esperada uma queda grande de S0 para S1. Isso não ocorreu. A evidência desta rodada é contrária a essa explicação simples. Blackhole e Wormhole permanecem mais difíceis, coerente com a necessidade de analisar contexto e confusões, embora o CSV atual não permita reconstrução temporal.


In [6]:
xgboost_by_class = pd.read_csv(report_dir / "xgboost_class_metrics.csv")
print(xgboost_by_class.to_string(index=False))


protocol       class_name  precision   recall       f1  support
      S0   Normal Traffic   0.989327 0.991709 0.990517    26172
      S0 Blackhole Attack   0.957112 0.860705 0.906352    26110
      S0  Flooding Attack   0.990577 0.985907 0.988236    19726
      S0     Sybil Attack   0.991232 0.995431 0.993327    24077
      S0  Wormhole Attack   0.862868 0.947481 0.903198    26086
      S1   Normal Traffic   0.989099 0.991556 0.990326    26172
      S1 Blackhole Attack   0.957224 0.860475 0.906275    26110
      S1  Flooding Attack   0.990440 0.976934 0.983641    19726
      S1     Sybil Attack   0.985402 0.995265 0.990309    24077
      S1  Wormhole Attack   0.861580 0.947520 0.902508    26086
      S2   Normal Traffic   0.989284 0.991174 0.990228    26172
      S2 Blackhole Attack   0.956568 0.861241 0.906405    26110
      S2  Flooding Attack   0.991110 0.966491 0.978646    19726
      S2     Sybil Attack   0.977172 0.995597 0.986299    24077
      S2  Wormhole Attack   0.861059 0.9

## Fronteira preliminar de custo

![Qualidade e tamanho](../../reports/predictive_baseline_v1/quality_vs_size_s2.png)

XGBoost domina esta rodada em F1 e tamanho frente às duas florestas. A MLP é menor, mas perdeu qualidade e não convergiu em 100 épocas. Os tempos registrados são diagnósticos amortizados e não respondem ainda à pergunta local versus borda.


## Decisões para a próxima rodada

- Manter XGBoost, RF e MLP compacta para tuning interno por grupo; Extra Trees pode permanecer como controle de alta memória.
- Implementar early stopping da MLP com uma divisão interna compatível com S1/S2 e repetir sementes.
- Manter `FlowID` somente na reprodução diagnóstica.
- Analisar erros Blackhole/Wormhole e ablações de atributos usando exclusivamente desenvolvimento na seleção.
- Não executar GNN ou análise temporal com o CSV atual.
- Selecionar e congelar modelos depois do tuning, antes do benchmark Docker/`tc-netem`.

Esta rodada não sustenta ainda uma afirmação final de superioridade. Ela estabelece que o protocolo funciona, que `FlowID` é um confundidor de grande magnitude e que XGBoost é o candidato inicial mais forte na fronteira qualidade–tamanho.
